# Reproducing Liu, Huang, Shen, Shao, Jing et al. (2026) from dandiset 001746

Paper: Liu, Huang, Shen, Shao, Jing et al. (2026). An atlas of primate insular cortex reveals a
signal-processing strategy in von Economo neurons. Nature Cell Biology 28, 1742-1757.
[doi:10.1038/s41556-026-02009-4](https://doi.org/10.1038/s41556-026-02009-4). The claims file was
written from the version of record in Europe PMC (PMC13457102): main text, legends, Methods; it
names the Supplementary Information as not read. This attempt also had the Supplementary
Information PDF (Supplementary Figs. 1 to 17); the Supplementary Tables are not in it.

Dataset: [DANDI:001746](https://dandiarchive.org/dandiset/001746/0.260825.0222), version
0.260825.0222, 462 NWB files, one Patch-seq cell each (current-clamp step sweeps). The paper
cites no version of the dandiset. This is the only published version, and the draft lists the
same 462 files and size (printed below), so it is pinned.

Outside inputs. The 10x excitatory neurons with their clusters, Ext.h5ad of the h5ad archive of
GEO GSE319557, streamed for fig7c. From the paper's code repository
[RFLiu2021/AIC_proj](https://github.com/RFLiu2021/AIC_proj) at commit
44cd8a6dfbb1f1ce363d73742e4f3652086b0f9d: the cell table metaInfo_ephysio_patchSeq.csv (cell
type, VEN subtype and transcriptomic cluster per file, which the files lack), the per-cell
feature table Ins_patchseq_ephs.xlsx and per-cell phase peak indices PeakIdx_Fig5e.xlsx (reference
values for checks that do not decide), NeuronPeakIdx_summary.mat (the 62 AIS distances of Fig.
4i), and the morphological feature table morphFeatMatrix.xlsx. Feature extraction follows the
Methods, with details the Methods leave open taken from the paper's MATLAB package PatchAnaLab 1.0
(doi:10.5281/zenodo.19995355), as the helper `aic.py` says for each.

| Claim | Finding | Tested or blocked |
|---|---|---|
| `patchseq-cell-table-link` | The recordings are the cells of the code's cell table, and their features match its feature table | Tested |
| `morph-table-link` | The code's morphology table matches the dandiset's cells (link claim added here) | Tested |
| `patchseq-sample` | 462 cells with intrinsic properties, 12 male macaques | Tested |
| `patchseq-mapping` | 307 typed cells, 273 excitatory in 18 types, 34 inhibitory in 17 types | Tested |
| `fig2a-subclass-counts` | Excitatory cells by subclass | Tested |
| `ven-map-et-ct` | 29 VENs map to L5 ET (14) and L5/6 CT (15) | Tested |
| `fig6f-ven-subtype-mapping` | VEN-L map to L5 ET (10), VEN-S to L5/6 CT (11) | Tested |
| `fig2d-l23it-small-tau` | L2/3 IT HPCAL1 have the smallest tau | Tested |
| `fig2f-l23it-narrow-ap` | L2/3 IT HPCAL1 have narrow APs | Tested |
| `supp11-rm-itrorb-l6pc` | IT RORB and L6 PC have a distinctive input resistance | Tested |
| `supp11-l5it-threshold-latency` | L5 IT FEZF2 have a higher threshold and longer latency | Tested |
| `supp11-ven-broad-ap` | VENs have the broadest APs | Tested |
| `supp11-l5et-deep-sag` | L5 ET have the deepest sag | Blocked: methods underspecified |
| `ven-firing-rate-variability` | VENs fire less and more irregularly than PCs | Tested |
| `l5et-vs-l5it-sag-rm-rate` | L5 ET against L5 IT: deeper sag, lower input resistance, higher rate | Tested |
| `fig3e-phase-peak-index` | VENs have the lowest phase peak index | Tested |
| `fig3d-saddle-phase-plane` | VEN phase plots are saddle-shaped | Tested |
| `fig4ac-ven-broader-smaller-ap` | 281 VENs against 306 L5 PCs: broader, smaller APs | Blocked: data missing |
| `fig4i-peak-index-ais-distance-real` | Peak index falls with AIS distance (62 neurons) | Tested |
| `ext9cd-ven-subtypes-ap` | VEN-L (85) and VEN-S (41) APs do not differ | Blocked: data missing |
| `supp8-ven-subtypes-similar-features` | VEN subtypes recorded without RNA do not differ | Blocked: data missing |
| `supp9-venl-more-distinct-from-ct` | VEN-L differ more from L5/6 CT than VEN-S do | Tested |
| `morph-more-distinct-than-ephys` | Morphology differs on more features than physiology | Tested |
| `fig7c-tau-rna-distance` | Tau distances correlate with transcriptomic distances | Tested |
| `scrna-classes`, `ext1d-ven-soma-density`, `fig3c-ais-distance-length`, `fig4d-sodium-current-lower`, `model-ais-distance-ap-shape`, `fig4j-eepsp-ven-larger`, `fig5-psc-and-connectivity`, `fig6-ven-subtype-morphology-markers`, `fig8-metabolomics` | Data in other archives (DANDI:001750 to 001752, GEO, MetaboLights, NeuroMorpho, the article's Source Data) | Blocked: other |

The `elsewhere` claims are tested under the claims files of the other dandisets (DANDI:001750 to
001752) and their criteria point there; this dandiset holds none of their data. The notebook
streams all 462 files (about 15 GB, read whole into memory one at a time per worker) and the first
5 GB of the GEO archive, and takes about seven minutes.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scikit_posthocs as sp
from scipy import stats
from scipy.io import loadmat

import aic

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
versions = requests.get(f"{aic.API}/dandisets/{aic.DANDISET_ID}/versions/", timeout=60).json()["results"]
info = requests.get(f"{aic.API}/dandisets/{aic.DANDISET_ID}/versions/{aic.DANDISET_VERSION}/", timeout=60).json()
assets = aic.list_assets()
cells = aic.load_all(assets)
print(f"read and measured {len(cells)} files of dandiset {aic.DANDISET_ID} version {aic.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s; extraction errors: {int(cells.get('error', pd.Series(dtype=str)).notna().sum())}")

# The paper's code tables (github.com/RFLiu2021/AIC_proj at the commit the claims file read).
meta = pd.read_csv(aic.META_CSV)
xfeat = pd.read_excel(aic.FEATURE_XLSX)
print("metaInfo_ephysio_patchSeq.csv:", meta.shape, "; Ins_patchseq_ephs.xlsx:", xfeat.shape)

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
print("Published versions:", [(v["version"], v["asset_count"], v["size"]) for v in versions])
s = info["assetsSummary"]
print("Archive summary: files", s["numberOfFiles"], "; subjects", s["numberOfSubjects"],
      "; species", [x["name"] for x in s["species"]], "; variables", s["variableMeasured"])
print("Files in the version's asset list:", len(assets), "; size", sum(a["size"] for a in assets) / 1e9, "GB")
print("\nFiles per subject:")
print(cells.groupby("subject_id").agg(files=("filename", "size"), sessions=("session_id", "nunique"),
                                       sex=("sex", lambda x: ",".join(sorted(set(x))))).to_string())
print("\nDistinct subjects:", cells.subject_id.nunique(), "; sessions:", cells.session_id.nunique(),
      "; session_start_time values:", cells.session_start.unique().tolist())
print("Sessions whose date is not the subject's date:")
d = cells.assign(sdate=cells.subject_id.str[3:], sess=cells.session_id.str[5:11])
print(d[d.sdate != d.sess].groupby(["subject_id", "session_id"]).size().to_string())
print("\nCell IDs unique:", cells.cell_id.is_unique, "; file identifier equals cell_id:", bool((cells.identifier == cells.cell_id).all()))

In [ ]:
print("Device against sampling rate and samples per sweep:")
print(pd.crosstab([cells.device, cells.fs.round(0)], cells.n_samples).to_string())
print("\nUnits stated: voltage", cells.v_unit.unique().tolist(), "; current", cells.i_unit.unique().tolist())
print("Voltage range over all sweeps (mV): min", cells.v_min.min().round(1), "max", cells.v_max.max().round(1),
      "; median per-file range", cells.v_min.median().round(1), "to", cells.v_max.median().round(1))
print("Current values: files by first step (pA)", cells.i_min.value_counts().to_dict(), "; maximum from", cells.i_max.min(), "to",
      cells.i_max.max(), "; increments", sorted(cells.i_step.unique()))
print("Sweeps per file:", cells.n_sweeps_file.value_counts().sort_index().to_dict())
print("Sweep numbers 1..n in order:", bool(cells.sweep_numbers_ok.all()))
print("Step duration (ms):", cells.step_ms.round(2).value_counts().to_dict())
print("Stimulus starting_time values:", cells.i_start.unique().tolist(), "; voltage starting_time:",
      cells.v_start.round(4).unique().tolist())
print("Resting potential (mV): median", cells.RMP.median().round(1), "range", cells.RMP.min().round(1), "to", cells.RMP.max().round(1))

The current series are stated in amperes; their values run from -200 to 1,480 in steps of 20, which
are pA (the Methods' steps from -200 pA in 20 pA increments). The voltage is in volts as stated
(resting potentials near -0.068 V, printed in mV above). Most files start at -200 pA (counts above),
the others at -120 to -60 pA, and the number of sweeps per file varies from 9 to 80, so the largest
step varies between files. Steps last 600 ms, as stated. The device names do not always fit the sampling rate: four
files named `AXON recording system` are sampled at 25 kHz like the HEKA files, and one `Heka` file
at 50 kHz. Every file has the same placeholder session start (2018-01-01 01:00), and some session
IDs carry the date of another subject. The subject IDs are taken as given.

In [ ]:
print("Clock check. Mean voltage change (mV) from the 10 ms before to the 10 ms after a time, on the largest")
print("hyperpolarizing sweep, at the stimulus onset, at the stimulus offset, and 100 ms before the onset:")
print(cells[["resp_at_onset", "resp_at_offset", "resp_100ms_before"]].describe().round(2).to_string())
print("Files with a hyperpolarizing response at the onset:", int((cells.resp_at_onset < 0).sum()), "of", len(cells),
      "; with a depolarizing return at the offset:", int((cells.resp_at_offset > 0).sum()))
print("Files where the change 100 ms before the onset exceeds half of that at the onset:",
      int((cells.resp_100ms_before.abs() > 0.5 * cells.resp_at_onset.abs()).sum()))
dd = cells.code_on_minus_stim_on.dropna()
print("Onset found in the voltage as the paper's code does (findStimPeriod.m) minus the stimulus onset, samples:"
      f" median {dd.median():.0f}; within 10 samples in {int((dd.abs() <= 10).sum())} of {len(dd)} files")

In every file the voltage falls at the stimulus onset and rises at its offset, so the voltage and
stimulus series share one clock. In some files a change of similar size appears 100 ms before the
onset (counted above); in the 600 ms protocol this is outside the step and is consistent with a
test pulse or spontaneous activity before it. It does not move the step. The paper's code finds the
step in the voltage within fixed sample windows; its onset lies within a few samples of the
stimulus onset in most files, and the analysis uses the stimulus onset.

In [ ]:
dup = cells[cells.trace_hash.duplicated(keep=False)].merge(
    meta.drop_duplicates("filename_nwb"), left_on="filename", right_on="filename_nwb", how="left")
print("Files with identical voltage data (all sweeps), by pair:")
print(dup.sort_values("trace_hash")[["trace_hash", "filename", "cell_id", "subject_id_x", "cellType", "cellSubtype",
                                     "predictCluster", "predictCluster_label"]].to_string(index=False))
print("Pairs:", dup.trace_hash.nunique(), "; distinct recordings in the dandiset:", cells.trace_hash.nunique())

Five pairs of files hold identical voltage data in every sweep under different cell IDs, four of
them under different subjects. Each pair is one recording deposited twice. In four pairs only
one of the two cells has a transcriptomic type in the code's table, so the other cannot enter an
analysis by type; in the fifth (B289 and n175) both have one, cluster 16 (PC, so L5 ET) and
cluster 8 (PC, the L5/6 CT PCs, in none of the six Fig. 2 groups). Counts below are given for files
and for distinct recordings. The copy n175 is left out of the L5/6 CT group of supp9-venl-more-distinct-from-ct, which
would otherwise hold a recording also filed as an L5 ET cell.

In [ ]:
paths = sorted(set().union(*cells.h5_paths))
print("HDF5 object paths over all files (digits as #):", len(paths))
print([p for p in paths if not p.startswith("specifications")])
strs = pd.DataFrame([(n, v) for t in cells.h5_strings for n, v in t], columns=["path", "value"])
print("\nString values and attributes outside the sweeps and specifications, distinct values per path:")
print(strs.groupby("path").value.nunique().to_string())
keys = r"VEN|PC_|Exc|Inh|INTER|cluster|FEZF2|RORB|HPCAL1|cell.?type|transcript"
hits = strs[strs.value.str.contains(keys, case=False, regex=True)]
print("\nValues mentioning a cell type, cluster or transcriptomic label:", len(hits))
print(hits.drop_duplicates().head(20).to_string(index=False))

Beyond the fields `aic.read_nwb` reads, the files hold no table, processing module or
annotation: the object tree above is the same in every file, and no string value or attribute
names a cell type, VEN subtype or transcriptomic cluster.

## patchseq-cell-table-link

> The dandiset's recordings are the Patch-seq cells of the paper's code table
> metaInfo_ephysio_patchSeq.csv, with the same subjects, expert cell types and predicted
> transcriptomic clusters, and the features computed from them agree with the paper's per-cell
> feature table.

Matched by file name. The files carry a subject and a cell ID, and no cell type or
transcriptomic type, so those are the labels compared. Features are computed as the Methods
define them (see the helper's docstring); the code's per-cell table is the reference.

In [ ]:
link = meta.merge(cells[["filename", "cell_id", "subject_id"]], left_on="filename_nwb", right_on="filename", how="left")
found = link.filename.notna()
print("Rows of the table:", len(meta), "; rows whose file is in the dandiset:", int(found.sum()),
      f"({found.mean():.1%})")
print("Rows naming a file another row also names:")
print(meta[meta.filename_nwb.duplicated(keep=False)].to_string())
print("Dandiset files named by no row:", sorted(set(cells.filename) - set(meta.filename_nwb)))
print("Rows whose cell ID agrees with the file's:", int((link.cell_ID == link.cell_id).sum()),
      "; whose subject agrees:", int((link.subject_id_x == link.subject_id_y).sum()))
print(link[(link.subject_id_x != link.subject_id_y)][["subject_id_x", "cell_ID", "filename_nwb", "subject_id_y", "cell_id"]].to_string())
unlisted = cells[~cells.filename.isin(meta.filename_nwb)][["filename", "cell_id", "subject_id"]]
print("The unlisted file:", unlisted.to_dict("records"))

In [ ]:
m1 = meta.drop_duplicates("cell_ID").set_index("cell_ID")
byfile = cells.set_index("filename")
xf = xfeat.copy()
xf["filename"] = xf.Cell_ID.map(m1.filename_nwb)
print("Rows of Ins_patchseq_ephs.xlsx:", len(xf), "; with a cell ID in the table:", int(xf.filename.notna().sum()),
      "; cluster in xlsx equals the table's:", int((xf.predictCluster.values == xf.Cell_ID.map(m1.predictCluster).values).sum()))
xm = xf.dropna(subset=["filename"]).join(byfile, on="filename", lsuffix="_xlsx")
pairs = [("half_width", "Half_Width_ms", "AP half-width (ms)"), ("amplitude", "AP_Amplitute", "AP amplitude (mV)"),
         ("max_firing_rate", "MaxFiringRate", "maximal firing rate (Hz)")]
rows = []
for mine, theirs, name in pairs + [("half_width_code", "Half_Width_ms", "half-width, code's threshold window (variant)"),
                                   ("tau_code", "Tau_ms", "tau, first sweep (code)"), ("tau_methods", "Tau_ms", "tau, all hyperpolarizing sweeps (Methods)"),
                                   ("Rm_methods", "Rm_MO", "input resistance (Methods)"), ("Rm_code", "Rm_MO", "input resistance (code)"),
                                   ("threshold", "Threshold_mV", "AP threshold"), ("latency", "Spike_Latency_ms", "spike latency"),
                                   ("rheobase", "Rheobase_pA", "rheobase"), ("sag_ratio", "Sag_Ratio", "sag ratio"),
                                   ("RMP", "RMP_mV", "RMP"), ("ISI_CV", "ISI_CV_xlsx", "ISI CV"), ("ISI_Fano", "ISI_FanoFactor", "ISI Fano factor")]:
    a, b = xm[mine].astype(float), pd.to_numeric(xm[theirs], errors="coerce")
    ok = a.notna() & b.notna()
    r = (a[ok] / b[ok] - 1).abs()
    rows.append({"feature": name, "cells": int(ok.sum()), "within_10pct": round(float((r <= 0.1).mean()), 3),
                 "median_ratio": round(float((a[ok] / b[ok]).median()), 3), "spearman": round(stats.spearmanr(a[ok], b[ok])[0], 3)})
agree = pd.DataFrame(rows)
print(agree.to_string(index=False))
print("\nThe criterion's three features (first three rows) within 10% for at least 80% of matched cells:",
      bool((agree.within_10pct[:3] >= 0.8).all()))

In [ ]:
lab = meta.drop_duplicates("filename_nwb").merge(cells, left_on="filename_nwb", right_on="filename", suffixes=("_table", ""))
lab["group"] = [aic.code_group(c, t) for c, t in zip(lab.predictCluster, lab.cellType)]
xf["group"] = [aic.code_group(c, t) for c, t in zip(xf.predictCluster, xf.ExpertCellType)]
legend = pd.Series([30, 17, 119, 22, 16, 13], index=aic.GROUPS)
cnt = pd.DataFrame({"legend Fig. 2d": legend, "table labels, all files": lab.group.value_counts(),
                    "xlsx labels (code)": xf.group.value_counts()}).loc[aic.GROUPS]
print(cnt.to_string())
both = xf.dropna(subset=["filename"]).merge(lab[["filename", "group"]], on="filename", suffixes=("_xlsx", "_table"))
print("\nCells in the xlsx whose group differs between its labels and the table's:",
      int((both.group_xlsx.fillna("-") != both.group_table.fillna("-")).sum()))
print(pd.crosstab(both.group_xlsx.fillna("-"), both.group_table.fillna("-")).to_string())

**Result: link reproduced.** All 462 rows of the cell table name a file in the dandiset, and every
row's cell ID equals its file's. One file (cell s172, subject Mfa180730) is named by no row; its
subject's row (row 235) instead names cell n172 and the file of n172 (subject Mfa220428), which row
110 also names, so that row is the only one whose subject disagrees (461 of 462 agree). Row 235 is
read as an error of the table for the file of s172, which no row names: it is a second row for a
file that has its own row, and it carries no type. Under the letter of the criterion ("not
reproduced when ... labels disagree") that row would fail the link (see the attempt's
deviations). The files carry no cell type or transcriptomic type to compare, and no other
annotation (the object tree above). For the 276 cells of the feature table, AP half-width is within
10% of the table's value for 96.7% of cells, AP amplitude for 97.5% and maximal firing rate for
98.9%, all above the criterion's 80%; threshold agrees for 97.4%. The six Fig. 2 groups formed by the
code's lists hold 30, 21, 125, 29, 18 and 14 files (the xlsx labels give 30, 22, 122, 29, 17 and 13)
against the legend's 30, 17, 119, 22, 16 and 13; this does not decide.

## patchseq-sample

> Of 578 Patch-seq cells profiled across all layers of the AIC, 462 were characterized for their
> intrinsic membrane properties; the Patch-seq experiments used 12 male macaques.

In [ ]:
print("Files with a current-clamp step protocol:", len(cells), "(expected 462, tolerance 46)")
print("Distinct recordings (identical pairs counted once):", cells.trace_hash.nunique())
print("Distinct subjects:", cells.subject_id.nunique(), "(expected 12, tolerance 1.2); sexes:", cells.sex.unique().tolist())

**Result: sample partially reproduced.** 462 files hold a current-clamp step protocol (457 distinct recordings,
as five pairs are identical), within 462 plus or minus 46. They come from 10 subjects, outside
12 plus or minus 1.2, all male.**

## patchseq-mapping

> Of the Patch-seq cells with transcriptomes, 307 were analysed; 273 excitatory neurons map to 18 of
> the 20 excitatory types and 34 inhibitory neurons to 17 of the 33 inhibitory types.

In [ ]:
typed = lab[lab.predictCluster_label.notna()].copy()
print("Rows with a mapping rate:", int(lab.mappingRate.notna().sum()), "; of these below 30%:", int((lab.mappingRate < 0.3).sum()),
      "; mapping rate range", lab.mappingRate.min(), "to", lab.mappingRate.max())
typed = typed[typed.mappingRate >= 0.3]
cls = typed.predictCluster_label.str[:3]
res = pd.DataFrame({
    "measured, files": [len(typed), int((cls == "Exc").sum()), int((cls == "Inh").sum()),
                        typed[cls == "Exc"].predictCluster_label.nunique(), typed[cls == "Inh"].predictCluster_label.nunique()],
    "expected": [307, 273, 34, 18, 17], "tolerance": [31, 27, 3.4, 1.8, 1.7]},
    index=["with a transcriptomic type", "excitatory", "inhibitory", "excitatory types", "inhibitory types"])
uniq = typed.drop_duplicates("trace_hash")
res["measured, distinct recordings"] = [len(uniq), int((uniq.predictCluster_label.str[:3] == "Exc").sum()),
                                        int((uniq.predictCluster_label.str[:3] == "Inh").sum()),
                                        uniq[uniq.predictCluster_label.str[:3] == "Exc"].predictCluster_label.nunique(),
                                        uniq[uniq.predictCluster_label.str[:3] == "Inh"].predictCluster_label.nunique()]
res["within"] = (res["measured, files"] - res.expected).abs() <= res.tolerance
print(res.to_string())

**Result: mapping partially reproduced.** 278 files have a transcriptomic type (expected 307, within tolerance),
248 excitatory (273, within) in 17 types (18, within), and 30 inhibitory (34 plus or minus 3.4,
outside) in 17 types (17, within). No typed row has a mapping rate below 30% (the lowest is
0.40): the table holds only the cells kept after the mapping. Counting identical recordings once
leaves 247 excitatory cells.**

## fig2a-subclass-counts

> The 273 excitatory Patch-seq neurons fall in seven of the eight subclasses, L2/3 IT HPCAL1 (33), IT
> RORB (23), L5 IT FEZF2 (143), L5 ET (20), L5/6 CT (11), L6 IT (5) and L6b (9).

Reading (a), the code's cluster lists; VENs counted apart. Reading (b), the dataset's metadata,
does not exist: the files give no subclass.

In [ ]:
exc = typed[typed.predictCluster_label.str.startswith("Exc")].copy()
is_ven = exc.cellType.fillna("").str.upper().str.contains("VEN")
def subclass(c, ven):
    c = int(c)
    if ven:
        return "VEN"
    for g, cl in aic.CLUSTERS.items():
        if c in cl:
            return {"L6 PC": "L6 IT + L6b"}.get(g, g)
    return {16: "L5 ET", 8: "L5/6 CT"}.get(c, "other")
exc["subclass"] = [subclass(c, v) for c, v in zip(exc.predictCluster, is_ven)]
exp = pd.DataFrame({"expected": [33, 23, 143, 20, 11, 14], "tolerance": [3.3, 2.3, 14.3, 2, 1.1, 1.4]},
                   index=["L2/3 IT HPCAL1", "IT RORB", "L5 IT FEZF2", "L5 ET", "L5/6 CT", "L6 IT + L6b"])
exp["measured"] = exc.subclass.value_counts().reindex(exp.index).fillna(0).astype(int)
exp["within"] = (exp.measured - exp.expected).abs() <= exp.tolerance
print(exp.to_string())
print("VENs, counted apart:", int((exc.subclass == "VEN").sum()), "; excitatory cells in no subclass:", int((exc.subclass == "other").sum()))
print("Excitatory total:", len(exc), "; without VENs:", int((exc.subclass != "VEN").sum()))
print("\nVariant (does not decide), VENs counted in their cluster's subclass:")
print(pd.Series([subclass(c, False) for c in exc.predictCluster]).value_counts().reindex(exp.index).to_string())
print("\nClusters by subclass (cluster number, label, cells):")
print(exc.groupby(["subclass", "predictCluster", "predictCluster_label"]).size().to_string())

**Result: fig2a partially reproduced.** Under the code's cluster lists, with the 29 VENs counted apart, 5 of
the 6 judged counts are within tolerance: L2/3 IT HPCAL1 30 (33), IT RORB 21 (23), L5 ET 18 (20),
L5/6 CT 11 (11), and L6 IT with L6b 14 (14). L5 IT FEZF2 holds 125, outside 143 plus or minus
14.3. The excitatory cells total 248 against the stated 273. Reading (b) cannot be run, as the
files give no subclass.**

## ven-map-et-ct

> The 29 morphologically identified VENs do not map to one transcriptomic cluster; 14 map to L5 ET
> and 15 to L5/6 CT.

In [ ]:
ven = typed[typed.cellType.fillna("").str.upper().str.contains("VEN")]
print("VENs (expert type VEN) with a transcriptomic type:", len(ven), "; distinct recordings:", ven.trace_hash.nunique())
print(ven.predictCluster_label.value_counts().to_string())
print("Expected 14 (tolerance 1.4) in Exc FEZF2 DSG2 and 15 (1.5) in Exc FEZF2 COL24A1")
print("All VENs in the table, with or without a type:", int(meta.drop_duplicates("filename_nwb").cellType.fillna("").str.upper().str.contains("VEN").sum()))

**Result: venmap reproduced.** The 29 VENs with a transcriptomic type map to Exc FEZF2 DSG2 (14) and Exc FEZF2
COL24A1 (15), and to no other cluster. The table holds 68 VENs in all.**

## fig6f-ven-subtype-mapping

> VENs mapped to L5 ET (Exc FEZF2 DSG2) are VEN-Ls (n = 10) and those mapped to L5/6 CT (Exc FEZF2
> COL24A1) are VEN-Ss (n = 11).

In [ ]:
sub = ven.cellSubtype.fillna("no subtype").str.replace("VEN_short", "VEN_Short")
ct = pd.crosstab(ven.predictCluster_label, sub)
print(ct.to_string())
print("Expected VEN_Long in Exc FEZF2 DSG2: 10 (tolerance 1); VEN_Short in Exc FEZF2 COL24A1: 11 (1.1)")
print("Subtype spellings in the table:", sorted(meta.cellSubtype.dropna().unique()))
print("VEN subtypes, all files:", meta.drop_duplicates("filename_nwb").cellSubtype.value_counts().filter(like="VEN").to_dict())

**Result: fig6f partially reproduced.** The subtypes follow the clusters as stated: every VEN_Long with a type
is in Exc FEZF2 DSG2 and every VEN_Short in Exc FEZF2 COL24A1, and none crosses. The counts are 12
VEN-L (expected 10 plus or minus 1, outside) and 10 VEN-S (11 plus or minus 1.1, within). The
other VENs of the two clusters carry a PC subtype label (L23_PC, PC_L23, PC_L5) or none.**

## Feature comparisons between the six projection neuron groups

The groups are formed as calFDR_ephsio.m forms them, from the table's cluster and expert type,
on the files with a transcriptomic type. The tests are two-sided Wilcoxon rank-sum tests with
Benjamini-Hochberg FDR over the features of each comparison (calFDR_ephsio.m). The claims file
makes the family the 39 features of Ins_patchseq_ephs.xlsx, those the Methods define computed as
defined there. Nineteen of them (phase areas and perimeters, ISI phase ratios, slopes of FR and
first ISI, amplitude CV and Fano factor, AHP latency and others) are defined neither in the
Methods nor in PatchAnaLab, so they cannot be computed from the recordings; their values are
taken from the code's table by cell ID (234 of the 237 grouped cells are in it), and the FDR over
the 39 features (`q39`) decides. The 20 features the Methods name form the variant family (`q_F20`;
for the ISI CV and Fano factor, which are not among them, these 20 and the two ISI measures).
`q39_upper` is the bound on the FDR over 39 features that holds whatever the 19 other p-values
are, shown as a check.

In [ ]:
grp = lab[lab.predictCluster_label.notna() & (lab.mappingRate >= 0.3) & lab.group.notna()].copy()
print("Cells per group (files):", grp.group.value_counts().reindex(aic.GROUPS).to_dict(), "; legend:", legend.to_dict())
print("Cells per group per subject:")
print(pd.crosstab(grp.subject_id, grp.group)[aic.GROUPS].to_string())
F20 = ["rheobase", "threshold", "amplitude", "rise", "decay", "half_width", "latency", "AHP", "max_firing_rate",
       "sag_value", "sag_ratio", "sag_time", "sag_area", "RMP", "Rm_methods", "tau_methods", "rebound",
       "ISI_adaptation", "amp_adaptation", "latency_slope"]
# The computed features that are among the 39 of the xlsx (max spike count and maximal rate are both there).
IN39 = ["tau_methods", "sag_ratio", "Rm_methods", "RMP", "rheobase", "latency", "threshold", "amplitude", "AHP", "rise",
        "decay", "half_width", "max_spikes", "max_firing_rate", "rebound", "ISI_adaptation", "amp_adaptation", "ISI_CV",
        "ISI_Fano", "latency_slope"]
EXTRA = ["ISI_CV", "ISI_Fano", "tau_code", "Rm_code", "max_spikes", "tau_sweep_median"]
# The 19 features of the xlsx defined neither in the Methods nor in PatchAnaLab, taken from the xlsx by cell ID
# (the code's own values), so that the BH family of 39 features can be formed.
X19 = ["ReboundSpike", "aveAPwidth_maxSweep", "RisDecayTRatio", "AHP_latency", "AP_widthRatio", "amp_ratio",
       "amp_FanoFactor", "amp_CV", "ISI_ratio", "ISI_phaseT_ratio", "ISI_phaseT_ratio_CV", "FR_slope", "firstISI_slope",
       "ISI_adpIdx_slope", "ampAdpIdx_slope", "firstSpikePhaseArea", "firstSpikePhasePerimeter", "spikePhaseAreaRatio",
       "spikePhasePerimeterRatio"]
assert len(IN39) + len(X19) == 39 and set(X19) <= set(xfeat.columns)
xv = xfeat.set_index(xfeat.Cell_ID.astype(str))[X19].apply(pd.to_numeric, errors="coerce")
for d, key in ((lab, "cell_ID"), (grp, "cell_ID"), (cells, "cell_id")):
    for f in X19:
        d[f] = d[key].astype(str).map(xv[f])
print("Grouped cells with the xlsx's values for the 19 features:", int(grp[X19].notna().any(axis=1).sum()), "of", len(grp))

def upper39(p):
    """Upper bound of the BH-adjusted p-value of each known feature in a 39-feature family."""
    p = pd.Series(p).dropna()
    r = p.rank(method="first")
    bound = (p * 39 / r).clip(upper=1)
    return pd.Series({k: bound[p >= p[k]].min() for k in p.index})

def compare(df, a_groups, b_groups, features=None):
    """Rank-sum tests of a against b for every feature, with the FDR readings described above."""
    a, b = df[df.group.isin(a_groups)], df[df.group.isin(b_groups)]
    feats = sorted(set(F20 + IN39 + EXTRA + X19), key=(F20 + IN39 + EXTRA + X19).index)
    rows = {}
    for f in feats:
        x, y = a[f].astype(float).dropna(), b[f].astype(float).dropna()
        p = stats.mannwhitneyu(x, y, alternative="two-sided").pvalue if len(x) > 1 and len(y) > 1 else np.nan
        rows[f] = {"n_a": len(x), "n_b": len(y), "median_a": x.median(), "median_b": y.median(), "p": p}
    t = pd.DataFrame(rows).T
    t["q_F20"] = np.nan
    t.loc[F20, "q_F20"] = aic.bh(t.loc[F20, "p"].astype(float))
    q22 = aic.bh(t.loc[F20 + ["ISI_CV", "ISI_Fano"], "p"].astype(float))
    t.loc[["ISI_CV", "ISI_Fano"], "q_F20"] = q22[-2:]
    t["q39_upper"] = upper39(t.loc[IN39, "p"].astype(float))
    t["q39"] = np.nan
    t.loc[IN39 + X19, "q39"] = aic.bh(t.loc[IN39 + X19, "p"].astype(float))
    for var, base in {"tau_code": "tau_methods", "Rm_code": "Rm_methods", "tau_sweep_median": "tau_methods"}.items():  # a variant in its base's place
        t.loc[var, "q39_upper"] = upper39(t.loc[[var if f == base else f for f in IN39], "p"].astype(float))[var]
        fam = [var if f == base else f for f in IN39 + X19]
        t.loc[var, "q39"] = aic.bh(t.loc[fam, "p"].astype(float))[fam.index(var)]
        fam = [var if f == base else f for f in F20]
        t.loc[var, "q_F20"] = aic.bh(t.loc[fam, "p"].astype(float))[fam.index(var)]
    t["F39"] = np.where(t.q39 < 0.05, "sig", "ns")
    t.loc[~t.index.isin(IN39 + ["tau_code", "Rm_code", "tau_sweep_median"]), "F39"] = "-"
    if features is not None:
        t = t.loc[features]
    return t

def readings(df, target, features):
    """Reading (a): target against the other five pooled; (b): against each other group."""
    others = [g for g in aic.GROUPS if g != target]
    out = []
    t = compare(df, [target], others, features)
    t.insert(0, "against", "other five pooled")
    out.append(t)
    for g in others:
        t = compare(df, [target], [g], features)
        t.insert(0, "against", g)
        out.append(t)
    r = pd.concat(out)
    r.index.name = "feature"
    return r.reset_index()

def medians(df, features):
    return df.groupby("group")[features].median().reindex(aic.GROUPS)

def show(r):
    cols = ["feature", "against", "n_a", "n_b", "median_a", "median_b", "p", "q39", "q_F20", "q39_upper", "F39"]
    print(r[cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

print("\nGroup medians of the measures used below:")
print(medians(grp, ["tau_methods", "tau_code", "half_width", "Rm_methods", "threshold", "latency", "sag_ratio",
                    "sag_value", "max_firing_rate", "ISI_CV", "ISI_Fano"]).round(3).to_string())
print("\nFeature values missing (no AP, too few APs, failed fit), cells:")
print(grp[F20 + ["ISI_CV", "ISI_Fano", "tau_code"]].isna().sum().to_string())

In [ ]:
print("Device by recording status, all files (typed: a transcriptomic type in the cell table):")
lab["typed"] = np.where(lab.predictCluster_label.notna(), "typed", "untyped")
lab["year"] = "20" + lab.subject_id.str[3:5]
print(pd.crosstab([lab.device, lab.typed], lab.year).to_string())
print("\nDevice by group (grouped cells):")
print(pd.crosstab(grp.device, grp.group)[aic.GROUPS].to_string())
print("\nUntyped files by expert type and device:")
print(pd.crosstab(lab[lab.typed == "untyped"].cellType.fillna("none"), lab[lab.typed == "untyped"].device).to_string())
print("\nMedians by group and device (median, then cells), for the measures that may depend on the sampling rate:")
for f in ["half_width", "phase_idx_first", "phase_idx_median", "tau_methods"]:
    t = grp.groupby(["group", "device"])[f].agg(["median", "size"]).unstack("device")
    print(f"{f}:"); print(t.reindex(aic.GROUPS).round(3).to_string())
print("\nThe code's own values (Ins_patchseq_ephs.xlsx), medians by device over the grouped cells it holds:")
xd = grp[["cell_ID", "device", "group"]].merge(xfeat.assign(cell_ID=xfeat.Cell_ID.astype(str)), on="cell_ID")
print(xd.groupby("device")[["Half_Width_ms", "Tau_ms", "AP_Amplitute", "Rm_MO"]].median().round(2).to_string())
print("\nRank-sum of a target group against the others pooled, within the files of one device:")
for target, f in [("L2/3 IT HPCAL1", "half_width"), ("L2/3 IT HPCAL1", "tau_methods"), ("VEN", "phase_idx_first")]:
    for dv in sorted(grp.device.unique()):
        d = grp[grp.device == dv]
        a, b = d[d.group == target][f].dropna(), d[d.group != target][f].dropna()
        p = stats.mannwhitneyu(a, b).pvalue if len(a) > 1 and len(b) > 1 else np.nan
        print(f"  {target} {f}, {dv}: n {len(a)} against {len(b)}, medians {a.median():.3f} against {b.median():.3f}, p {p:.3g}")

# Is the device difference one of sampling rate? 40 grouped Axon files at 50 kHz, brought to 25 kHz (the HEKA rate) by
# averaging sample pairs, and measured again.
ax = grp[(grp.device.str.startswith("AXON")) & (grp.fs.round() == 50000)].sample(40, random_state=0)
aid = {a["path"].split("/")[-1]: a["asset_id"] for a in assets}
rows = []
for fn in ax.filename:
    rec = aic.read_nwb(aid[fn])
    n2 = rec["V"].shape[1] // 2 * 2
    rec["V"] = rec["V"][:, :n2].reshape(len(rec["V"]), -1, 2).mean(axis=2)
    rec["I"] = rec["I"][:, :n2:2]
    rec["fs"] = rec["fs"] / 2
    f25 = aic.cell_features(rec)
    rows.append({"filename": fn, **{k + "_25k": f25[k] for k in ["half_width", "tau_methods", "phase_idx_first", "amplitude"]}})
dec = ax.merge(pd.DataFrame(rows), on="filename")
for k in ["half_width", "tau_methods", "phase_idx_first", "amplitude"]:
    print(f"{k}: median at 50 kHz {dec[k].median():.3f}, decimated to 25 kHz {dec[k + '_25k'].median():.3f}; "
          f"median ratio {(dec[k + '_25k'] / dec[k]).median():.3f} (40 Axon files)")

The Methods put the recordings without RNA on a MultiClamp 700B with pClamp and the Patch-seq
recordings on a HEKA amplifier with PatchMaster ("Patch-seq recording"). In the files the device
follows the recording year (mostly Axon for the 2018 subjects and HEKA for the 2022 ones), and both
devices hold typed and untyped cells, so the device does not mark the recordings without RNA. The
groups are unevenly spread over the devices (L5 IT FEZF2 93 of 125 Axon files, L2/3 IT HPCAL1 6 of
30). Half-width and tau are longer in the Axon files in every group, in the computed values and in
the code's own table (half-width 3.13 against 2.08 ms, tau 41 against 26 ms). Bringing Axon files to
the HEKA rate leaves half-width and tau unchanged and changes the phase peak index, so the
half-width and tau difference is one of recording conditions, not of sampling rate, while the
phase peak index also depends on the rate.

## fig2d-l23it-small-tau

> Among the six projection neuron groups, L2/3 IT HPCAL1 neurons stand out with a small membrane
> time constant.

In [ ]:
med = medians(grp, ["tau_methods", "tau_code"])
print("Median tau by group (ms):"); print(med.round(2).to_string())
print("Lowest median, Methods tau:", med.tau_methods.idxmin(), "; first-sweep tau (code):", med.tau_code.idxmin())
r = readings(grp, "L2/3 IT HPCAL1", ["tau_methods", "tau_code"])
show(r)
g2 = grp[~(grp.tau_code <= 1)]
print("\nVariant, the code's plot exclusion (tau of 1 ms or less), cells dropped:", int((grp.tau_code <= 1).sum()))
print("Per subject, median tau of L2/3 IT HPCAL1 and of the others pooled (Methods tau):")
print(grp.assign(t=np.where(grp.group == "L2/3 IT HPCAL1", "L2/3 IT", "others")).pivot_table(
    index="subject_id", columns="t", values="tau_methods", aggfunc=["median", "count"]).round(2).to_string())
print("\nFits per cell for the Methods' tau (hyperpolarizing sweeps fitted, of those there):")
print(grp.groupby("group").agg(fits_median=("tau_n_fits", "median"), fits_min=("tau_n_fits", "min"),
                               sweeps_median=("tau_n_sweeps", "median")).reindex(aic.GROUPS).to_string())
spread = grp.tau_sweep_max / grp.tau_sweep_min
print("Spread of tau across a cell's sweeps (max over min): median", round(spread.median(), 2), "; above 2 in",
      int((spread > 2).sum()), "of", len(grp), "cells")
print("Cells with a Methods tau above 150 ms:", grp[grp.tau_methods > 150][["cell_ID", "group", "tau_methods", "tau_sweep_median",
      "tau_sweep_min", "tau_sweep_max", "tau_n_fits", "tau_code"]].round(1).to_string(index=False))
print("\nVariant, the median over sweeps in place of the mean (robust to single poor fits):")
print(medians(grp, ["tau_sweep_median"]).round(2).T.to_string())
show(readings(grp, "L2/3 IT HPCAL1", ["tau_sweep_median"]).iloc[:1])

**Result: fig2d reproduced** L2/3 IT HPCAL1 has the lowest median tau of the six
groups with the Methods' tau (32.7 ms; next IT RORB, 41.0) and with the code's first-sweep tau
(24.2 ms). Against the other five pooled (reading a) p = 0.011, FDR 0.034 over the 39-feature
family (the 19 features defined nowhere taken from the code's table) and 0.023 over the 20 Methods
features. With the code's tau p = 0.0003, FDR 0.003; with the median over sweeps in place of the
mean, p = 0.004, FDR 0.017. Reading (b) fails: the Methods' tau does not differ significantly from
IT RORB, VEN, L5 ET or L6 PC. L2/3 IT HPCAL1 has the lower median in 4 of the 6 subjects with such
cells. Within one device the difference is not significant (Axon files, 6 cells, p = 0.47; HEKA
files, 24 cells, p = 0.13): 24 of the 30 L2/3 IT HPCAL1 cells are HEKA files, whose tau is shorter
in every group (data check above).

## fig2f-l23it-narrow-ap

> L2/3 IT HPCAL1 neurons have a narrow AP width compared with the other projection neuron groups.

In [ ]:
med = medians(grp, ["half_width", "half_width_code"])
print("Median half-width by group (ms):"); print(med.round(3).to_string())
print("Rank of L2/3 IT HPCAL1 (1 = lowest):", int(med.half_width.rank()["L2/3 IT HPCAL1"]))
r = readings(grp, "L2/3 IT HPCAL1", ["half_width"])
show(r)
g2 = grp[(grp.half_width >= 1) & (grp.half_width <= 5)]
print("\nVariant, the code's plot exclusion (widths outside 1 to 5 ms), cells kept:", len(g2), "of", len(grp))
show(readings(g2, "L2/3 IT HPCAL1", ["half_width"]).iloc[:1])

**Result: fig2f partially reproduced.** L2/3 IT HPCAL1 APs are narrower than those of the other five pooled
(median half-width 2.08 against 2.82 ms, p = 3.0e-5; FDR 0.0006 over 39 features, 0.0003 over 20),
and narrower than L5 IT FEZF2 and VEN, but not than IT RORB (1.92 ms, p = 0.61), L5 ET (2.22 ms,
p = 0.45) or L6 PC (2.34 ms, p = 0.30). Its median is the second lowest, so the criterion's "lowest"
fails while reading (a) is significant. With the code's plot exclusion (1 to 5 ms) the pooled
comparison holds (p = 6.4e-5). Within one device the difference is not significant (Axon p = 0.37,
HEKA p = 0.19); half-widths of HEKA files are shorter in every group.

## supp11-rm-itrorb-l6pc

> IT RORB and L6 PC neurons have a distinctive input resistance among projection neuron groups.

In [ ]:
print("Median input resistance by group (MOhm):"); print(medians(grp, ["Rm_methods", "Rm_code"]).round(1).to_string())
for target in ["IT RORB", "L6 PC"]:
    print(f"\n{target}:")
    show(readings(grp, target, ["Rm_methods", "Rm_code"]))

**Result: rm partially reproduced** L6 PC neurons have a distinctive, higher
input resistance: median 354 against 166 MOhm for the other five pooled, p = 0.00014, FDR 0.006
over 39 features and 0.003 over 20, and the same with the code's measure. IT RORB neurons have 217
against 166 MOhm, p = 0.023, FDR 0.087 over 39 features and 0.065 over 20, not significant; with the
code's measure (a variant) p = 0.0055, FDR 0.037. Against each group (reading b) neither holds for
all. One of the two groups holds; both have the higher resistance.

## supp11-l5it-threshold-latency

> L5 IT FEZF2 neurons have a higher AP threshold and a longer spike latency than the other
> projection neuron groups.

In [ ]:
print(medians(grp, ["threshold", "threshold_code", "latency"]).round(2).to_string())
show(readings(grp, "L5 IT FEZF2", ["threshold", "latency"]))

**Result: l5it reproduced.** Against the other five pooled, L5 IT FEZF2 neurons have a higher AP threshold
(median -39.1 against -42.3 mV, p = 0.0012, FDR 0.005 over 39 features) and a longer spike latency
(203.7 against 148.7 ms, p = 0.00016, FDR 0.0008), also below 0.05 over the 20 features. Against
each group (reading b) the threshold is not higher than that of L2/3 IT HPCAL1 (-38.2 mV) or VEN,
and the latency is not significantly longer than that of IT RORB, VEN, L5 ET or L6 PC.

## supp11-ven-broad-ap

> VENs have a broader AP than the other Patch-seq projection neuron groups.

In [ ]:
med = medians(grp, ["half_width"])
print(med.round(3).to_string()); print("Highest median:", med.half_width.idxmax())
show(readings(grp, "VEN", ["half_width"]))

**Result: venap partially reproduced.** The VEN median half-width (3.03 ms) is the highest of the six,
narrowly above L5 IT FEZF2 (2.98 ms). Against the other five pooled p = 0.035, FDR 0.25 over 39
features (0.23 over 20), not significant. Against each group (reading b) VENs are broader than
L2/3 IT HPCAL1, IT RORB, L5 ET and L6 PC (FDR 0.0003 to 0.029) but not than L5 IT FEZF2 (p = 0.92).
The median is highest without a significant reading.

## supp11-l5et-deep-sag

> L5 ET neurons have a deeper sag than the other projection neuron groups.

Deeper is read as a larger sag ratio and a larger sag value. Both are measured at each file's
largest hyperpolarizing step, -200 pA in most files; the variant keeps only those files.

In [ ]:
med = medians(grp, ["sag_ratio", "sag_value"])
print(med.round(3).to_string()); print("Largest median: sag ratio", med.sag_ratio.idxmax(), "; sag value", med.sag_value.idxmax())
show(readings(grp, "L5 ET", ["sag_ratio", "sag_value"]))
print("\nCells whose largest hyperpolarizing step is not -200 pA, by group:", grp[grp.i_min > -200].group.value_counts().to_dict())
print("\nVariant, cells whose largest hyperpolarizing step is -200 pA only (sag measured at the same current):")
g2 = grp[grp.sag_step_pA == -200]
print(medians(g2, ["sag_ratio", "sag_value"]).round(3).T.to_string())
show(readings(g2, "L5 ET", ["sag_ratio", "sag_value"]).iloc[:2])
show(compare(g2, ["L5 ET"], ["L5 IT FEZF2"], ["sag_ratio", "sag_value"]).assign(against="L5 IT FEZF2").reset_index(names="feature"))

**Result: sag blocked (methods underspecified).** The two readings of the measure disagree. By sag ratio
L5 ET neurons have the deepest sag (median 1.36 against 1.13 for the other five pooled, p = 3e-6,
FDR 0.0001 over 39 features). By sag value, which the Methods define as RMP minus the steady
state at the largest hyperpolarizing step, L5 ET has the smallest of the six groups (19.6 against
27.7 mV, p = 0.0055). On the cells whose largest step is -200 pA only, the same holds (sag ratio
1.36, p = 5e-7; sag value 20.6 against 29.2 mV, p = 0.011), so the mixed step sizes do not cause the
disagreement. The paper does not say which measure its "deeper sag" uses.

## ven-firing-rate-variability

> VENs have lower firing rates and greater variability in spike timing than PCs.

In [ ]:
feats = ["max_firing_rate", "ISI_CV", "ISI_Fano"]
print(medians(grp, feats).round(3).to_string())
print("\n(a) VEN against the five PC groups pooled, and (b) against each:")
show(readings(grp, "VEN", feats))
print("\n(c) VEN against the layer 5 PC groups pooled (L5 IT FEZF2 and L5 ET):")
t = compare(grp, ["VEN"], ["L5 IT FEZF2", "L5 ET"], feats).reset_index().rename(columns={"index": "feature"})
t.insert(1, "against", "L5 IT + L5 ET")
show(t)

**Result: venfr not reproduced.** VENs have the lowest median maximal firing rate (10.0 Hz against 13.3 Hz
for the five PC groups pooled), but not significantly (p = 0.092, FDR 0.30 over 39 features), nor
against the layer 5 PCs pooled (p = 0.38); against each group it is significant only for L2/3 IT
HPCAL1 and L5 ET. VENs have a higher ISI CV (0.33 against 0.31, p = 0.56) and Fano factor (9.6
against 7.5, p = 0.17) without significance under readings (a) and (c), and under (b) only the
Fano factor against L2/3 IT HPCAL1 (FDR 0.047). The two variability measures agree.

## l5et-vs-l5it-sag-rm-rate

> L5 ET neurons have a deeper sag, a lower input resistance and a higher firing rate than L5 IT
> neurons.

In [ ]:
t = compare(grp, ["L5 ET"], ["L5 IT FEZF2"], ["sag_ratio", "sag_value", "Rm_methods", "Rm_code", "max_firing_rate"])
t.insert(0, "against", "L5 IT FEZF2")
show(t.reset_index().rename(columns={"index": "feature"}))

**Result: l5et partially reproduced.** Against L5 IT FEZF2, L5 ET neurons have a larger sag ratio (1.36
against 1.13, p = 1.6e-6, FDR 6e-5 over 39 features), a lower input resistance (116 against 160
MOhm, p = 0.014, FDR 0.039; the code's measure 117 against 174 MOhm, FDR 0.020) and a higher maximal
firing rate (15.8 against 11.7 Hz, p = 0.0063, FDR 0.022). By sag value L5 ET is lower (19.6
against 27.1 mV, p = 0.015), the reverse of deeper. The criterion takes the two sag readings of
supp11-l5et-deep-sag: under the sag ratio all three hold, under the sag value two do. The outcome
is the one that holds under both readings, two of three.

## fig3e-phase-peak-index and fig3d-saddle-phase-plane

> VENs have lower phase peak indices than any PC group.

> The AP phase-plane plots of VENs show a more prominent biphasic shape than those of PCs, with the
> AIS phase decaying before the SD phase in a saddle-like shape.

The index is read automatically (helper docstring) on the maximal-rate sweep, (a) on its first AP
and (b) as the median over its APs. Kruskal-Wallis over the six groups, then Dunn's test with
Bonferroni and with Sidak correction. The code's per-cell values (PeakIdx_Fig5e.xlsx, read by
hand) are compared, without deciding.
An AP whose rising phase has neither a second maximum nor a shoulder gets the lowest dV/dt of
its rising phase as its AIS value (the Methods' "point with the lowest derivative value"), so it
counts as the least saddle-like; the synthetic APs below show the three cases, and the variant
`_x` leaves such APs out.

In [ ]:
def dunn_table(df, col):
    d = df.dropna(subset=[col])
    kw = stats.kruskal(*[d[d.group == g][col] for g in aic.GROUPS])
    out = {"KW H": kw.statistic, "KW p": kw.pvalue}
    for corr in ["bonferroni", "sidak"]:
        pt = sp.posthoc_dunn(d, val_col=col, group_col="group", p_adjust=corr)
        for g in aic.GROUPS:
            if g != "VEN":
                out[f"{corr} VEN vs {g}"] = pt.loc["VEN", g]
    return d.groupby("group")[col].agg(["count", "median"]).reindex(aic.GROUPS), out

# Synthetic APs: dV/dt with two maxima (biphasic), with a shoulder, and with a single smooth maximum (monophasic).
fs_s = 50000.0
ts = np.arange(0, 0.004, 1 / fs_s)
bump = lambda t0, w, h: h * np.exp(-0.5 * ((ts - t0) / w) ** 2)
synth = {"biphasic": bump(0.0012, 0.00015, 150) + bump(0.0018, 0.0002, 250),
         "shoulder": bump(0.0013, 0.0002, 120) + bump(0.0017, 0.0002, 250),
         "monophasic": bump(0.0016, 0.0003, 300)}
for name, dv in synth.items():  # dv in mV/ms
    v = -50 + np.cumsum(dv) / fs_s * 1000
    peak = int(np.argmax(dv)) + int(0.0004 * fs_s)
    print(f"Synthetic {name} AP: index {aic.phase_index(v, fs_s, int(0.0005 * fs_s), peak)}")

print("\nFirst AP of the maximal-rate sweep by how its AIS value was read (cells):")
print(pd.crosstab(grp.group, grp.phase_first_kind).reindex(aic.GROUPS).to_string())
print("Fraction of the sweep's APs with the AIS phase not resolved (no second maximum, no shoulder), median per cell and mean:")
print(grp.groupby("group").phase_none_frac.agg(["median", "mean"]).reindex(aic.GROUPS).round(3).to_string())

for col in ["phase_idx_first", "phase_idx_median", "phase_idx_first_x", "phase_idx_median_x",
            "phase_idx_first_p2", "phase_idx_median_p2", "phase_idx_first_p10", "phase_idx_median_p10"]:
    medtab, out = dunn_table(grp, col)
    print(f"\n{col}: medians by group"); print(medtab.round(3).T.to_string())
    print({k: float(f"{v:.3g}") for k, v in out.items()})
print("\nFraction of APs whose rising phase has two maxima (median per cell, by group):")
print(grp.groupby("group").phase_two_peaks_frac.median().reindex(aic.GROUPS).round(3).to_string())

In [ ]:
pk_raw = pd.read_excel(aic.PEAKIDX_XLSX, sheet_name="Sheet1")
code_n = {"L23IT_HAPCAL1": 30, "ITrorb": 22, "L5IT": 122, "VEN": 29, "L5ET": 17, "L6PC": 13}  # Fig3e_plotBoxplot_PeakIdx_cellTypes.m
rows = []
for i in range(0, 18, 3):
    name = pk_raw.columns[i]
    for c, v in zip(pk_raw.iloc[:code_n[name], i], pk_raw.iloc[:code_n[name], i + 1]):
        rows.append({"cell_ID": str(c).strip("'"), "code_group": name, "code_idx": v})
pkc = pd.DataFrame(rows)
pkc["filename"] = pkc.cell_ID.map(m1.filename_nwb)
pkc = pkc.join(byfile[["phase_idx_first", "phase_idx_median", "phase_idx_first_x", "phase_idx_median_x"]], on="filename")
print("Cells in the code's table:", len(pkc), "; found in the dandiset:", int(pkc.filename.notna().sum()))
for c in ["phase_idx_first", "phase_idx_median", "phase_idx_first_x", "phase_idx_median_x"]:
    ok = pkc[c].notna() & pkc.code_idx.notna()
    print(f"{c}: Spearman with the hand-read values {stats.spearmanr(pkc[c][ok], pkc.code_idx[ok])[0]:.2f}, "
          f"median absolute difference {(pkc[c] - pkc.code_idx).abs().median():.3f}, cells {int(ok.sum())}")
print("Code's values, medians by its groups:", pkc.groupby("code_group").code_idx.median().round(3).to_dict())
gmap = {"L23IT_HAPCAL1": "L2/3 IT HPCAL1", "ITrorb": "IT RORB", "L5IT": "L5 IT FEZF2", "VEN": "VEN", "L5ET": "L5 ET", "L6PC": "L6 PC"}
pkc["group"] = pkc.code_group.map(gmap)
medtab, out = dunn_table(pkc, "code_idx")
print("Kruskal-Wallis and Dunn on the code's per-cell values:", {k: float(f"{v:.3g}") for k, v in out.items()})

**Result: fig3e partially reproduced.** Both AP readings and both corrections give the same outcome. On
the first AP of the maximal-rate sweep the VEN median index is the lowest (0.037; next L2/3 IT
HPCAL1 0.083), Kruskal-Wallis p = 0.0016, and Dunn's test separates VENs from L5 ET (Bonferroni p =
0.003, Sidak 0.003) and from no other group (L6 PC 0.062). On the median over the sweep's APs the VEN
median is again the lowest (0.081; next L5 IT FEZF2 0.111), Kruskal-Wallis p = 0.0015, and Dunn's
test separates VENs from L5 ET only (p = 0.009 under both). Leaving out the few APs whose AIS phase
is not resolved (at most 3% of a group's APs) leaves the VEN medians the lowest (0.030 and
0.081) and separates VENs from IT RORB, L5 ET and L6 PC on the first AP. The code's hand-read values for 233 cells give the paper's
result (VEN median 0.117, the lowest; Dunn p of 0.001 or less against every PC group, both
corrections), and the automatic index agrees with them only loosely (Spearman 0.39 first AP, 0.20
median), so the difference from the paper's result rests on the automatic peak reading, which the
criterion reports as a deviation. The index depends on the sampling rate (Axon files brought to
25 kHz change it, median 0.088 to 0.168), and within one device VENs are not significantly lower
than the other groups pooled (Axon p = 0.08, HEKA p = 0.10).

In [ ]:
pc = grp[grp.group != "VEN"]
print("Description that does not decide: median fraction of APs (maximal-rate sweep) with two maxima of dV/dt on the rise")
print("VEN", round(grp[grp.group == "VEN"].phase_two_peaks_frac.median(), 3), "; PC groups pooled", round(pc.phase_two_peaks_frac.median(), 3),
      "; rank-sum p", f"{stats.mannwhitneyu(grp[grp.group == 'VEN'].phase_two_peaks_frac.dropna(), pc.phase_two_peaks_frac.dropna()).pvalue:.3g}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, col in zip(ax, ["phase_idx_first", "phase_idx_median"]):
    a.boxplot([grp[grp.group == g][col].dropna() for g in aic.GROUPS], tick_labels=[g.split()[0] for g in aic.GROUPS])
    a.set_title(col); a.set_ylabel("phase peak index")
plt.tight_layout(); plt.show()
print("Plotted ranges:", {c: (round(grp[c].min(), 3), round(grp[c].max(), 3)) for c in ["phase_idx_first", "phase_idx_median"]})

**Result: fig3d partially reproduced,** as fig3e-phase-peak-index, whose outcome it takes. As a
description that does not decide, the median fraction of APs with two maxima of dV/dt on the rise
is 0.80 in VENs against 0.60 in the PC groups pooled (p = 0.34).

## fig4ac-ven-broader-smaller-ap

> APs of VENs (n = 281) are broader and lower in amplitude than those of L5 PCs (n = 306), throughout
> the spike train evoked by supra-threshold current.

In [ ]:
allc = meta.drop_duplicates("filename_nwb").merge(cells, left_on="filename_nwb", right_on="filename", suffixes=("_table", ""))
v = allc[allc.cellType.fillna("").str.upper().str.contains("VEN")]
l5 = allc[allc.cellSubtype.isin(["PC_L5", "PC_L5ET"])]
print("Recordings with expert type VEN:", len(v), "; with L5 PC labels (PC_L5, PC_L5ET):", len(l5), "; expected 281 and 306")
for f in ["half_width", "amplitude", "penult_half_width", "penult_amplitude"]:
    x, y = v[f].dropna(), l5[f].dropna()
    print(f"{f}: VEN median {x.median():.3f} (n {len(x)}), L5 PC {y.median():.3f} (n {len(y)}), rank-sum p {stats.mannwhitneyu(x, y).pvalue:.3g}")

**Result: fig4ac blocked (data missing).** The dandiset holds 68 VENs and 138 cells labelled as L5 PCs (PC_L5,
PC_L5ET), the Patch-seq cells, against the 281 VENs and 306 L5 PCs of Fig. 4a-c, which the paper
calls additional recordings. The 113 Axon files without a transcriptomic type are rows of the
Patch-seq cell table and are described as Patch-seq in every file, so they are not taken as those
recordings. As a variant on the Patch-seq cells, which does not decide, VEN APs are broader (3.08
against 2.70 ms, p = 0.009) and smaller (86.2 against 90.2 mV, p = 0.071), neither with p below the
criterion's 0.001; at the penultimate AP of the maximal-rate sweep 5.10 against 4.72 ms (p = 0.060)
and 54.2 against 57.3 mV (p = 0.34).

## fig4i-peak-index-ais-distance-real

> In real neurons the phase peak index decreases linearly with the distance of the AIS from the soma
> (n = 62).

The 62 neurons are taken as the Patch-seq cells with a reconstructed axon origin (the criterion):
those with an AIS distance (axonDist) in morphFeatMatrix.xlsx, matched by cell ID. As in
Fig4itop_plot_PhasePeakIdx_vs_AISd.m, the indices are averaged in 10 um bins and a line is fitted
to the bin means against the bins' left edges, its slope tested by a two-tailed t-test.

In [ ]:
mat = loadmat(__import__("io").BytesIO(requests.get(aic.AIS_MAT, timeout=60).content))
print("NeuronPeakIdx_summary.mat holds:", {k: v.shape for k, v in mat.items() if not k.startswith("__")})
vals = mat["phaseHeightIdx"].ravel()
allpk = []
for i in range(0, 18, 3):
    allpk += list(pd.to_numeric(pk_raw.iloc[:, i + 1], errors="coerce").dropna())
allpk = np.array(allpk)
hits = [int(np.isclose(allpk, x, rtol=0, atol=1e-6).sum()) for x in vals]
print("Of its 62 peak indices, found in PeakIdx_Fig5e.xlsx (the per-cell table with cell IDs):", sum(h > 0 for h in hits))
mx0 = pd.read_excel(aic.MORPH_XLSX, sheet_name="morphFeatMatrix")
ad = mat["axonDist"].ravel()
print("Of its 62 AIS distances, nonzero:", int((ad > 0).sum()), "; nonzero ones found among the axonDist values of morphFeatMatrix.xlsx:",
      int(sum(np.isclose(mx0.axonDist.dropna(), a, rtol=0, atol=1e-3).any() for a in ad if a > 0)))

# The criterion's reading: the 62 neurons taken as the Patch-seq cells with a reconstructed axon origin, AIS distance
# from morphFeatMatrix.xlsx (axonDist, um) by cell ID.
mx0["cell_ID"] = mx0["Unnamed: 0"].astype(str).str.strip("'")
ais = lab.merge(mx0[["cell_ID", "axonDist"]], on="cell_ID").dropna(subset=["axonDist"])
ais = ais.join(pkc.drop_duplicates("cell_ID").set_index("cell_ID")[["code_idx"]], on="cell_ID")
print("\nPatch-seq cells with an AIS distance in morphFeatMatrix.xlsx:", len(ais), "(legend 62); by group:",
      ais.group.fillna("none").value_counts().to_dict(), "; distance 0:", int((ais.axonDist == 0).sum()),
      "; range", ais.axonDist.min().round(1), "to", ais.axonDist.max().round(1), "um")
fig4i = {}
for col, nz in [(x, False) for x in ["phase_idx_first", "phase_idx_median", "phase_idx_first_x", "phase_idx_median_x", "code_idx"]] + \
        [("phase_idx_first", True), ("phase_idx_median", True), ("code_idx", True)]:
    d = ais.dropna(subset=[col])
    d = d[d.axonDist > 0] if nz else d
    b = d.assign(bin=(d.axonDist // 10) * 10).groupby("bin")[col].agg(["mean", "size"])
    lr = stats.linregress(b.index, b["mean"])
    lc = stats.linregress(d.axonDist, d[col])
    fig4i[col + (", distance > 0 (variant)" if nz else "")] = {"cells": len(d), "bins": len(b), "slope per um, bins": lr.slope, "p, bins": lr.pvalue,
                  "slope per um, cells (variant)": lc.slope, "p, cells": lc.pvalue}
print(pd.DataFrame(fig4i).T.to_string(float_format=lambda v: f"{v:.4g}"))

**Result: fig4i not reproduced.** The 62 AIS distances of the paper's code carry no cell IDs and match
no per-cell table, so the criterion's reading is used: the Patch-seq cells with an AIS distance in
morphFeatMatrix.xlsx, 153 of them (106 at 0 um) where the legend gives 62. Over 10 um bins the line
through the bin means has a slope of +0.00005 per um (p = 0.92) for the first-AP index and +0.0003
(p = 0.62) for the median index, so neither reading gives the negative slope. The code's hand-read
indices on the same cells (127) give a negative slope (-0.0027 per um, p = 0.018 over 7 bins), so
the difference rests on the automatic peak reading, as for fig3e-phase-peak-index. Leaving out the
cells at 0 um does not change this.

## ext9cd-ven-subtypes-ap and supp8-ven-subtypes-similar-features

In [ ]:
vsub = allc[allc.cellType.fillna("").str.upper().str.contains("VEN")].copy()
vsub["sub"] = vsub.cellSubtype.str.replace("VEN_short", "VEN_Short")
print("VENs by expert subtype, all files:", vsub["sub"].value_counts(dropna=False).to_dict(), "; expected VEN-L 85 and VEN-S 41")
a, b = vsub[vsub["sub"] == "VEN_Long"], vsub[vsub["sub"] == "VEN_Short"]
p = [stats.mannwhitneyu(a[f].dropna(), b[f].dropna()).pvalue for f in ["half_width", "amplitude"]]
for f, pp, q in zip(["half_width", "amplitude"], p, aic.bh(p)):
    print(f"{f}: VEN-L median {a[f].median():.3f} (n {a[f].notna().sum()}), VEN-S {b[f].median():.3f} (n {b[f].notna().sum()}), "
          f"difference {a[f].median() - b[f].median():.3f}, p {pp:.3g}, FDR over the two {q:.3g}")

In [ ]:
def two_group(dfa, dfb, family=F20):
    rows = {f: {"median_a": dfa[f].median(), "median_b": dfb[f].median(), "n_a": dfa[f].notna().sum(), "n_b": dfb[f].notna().sum(),
                "p": stats.mannwhitneyu(dfa[f].dropna(), dfb[f].dropna()).pvalue if dfa[f].notna().sum() > 1 and dfb[f].notna().sum() > 1 else np.nan}
            for f in IN39 + X19 + [x for x in F20 if x not in IN39]}
    t = pd.DataFrame(rows).T
    t["q_F20"] = np.nan
    t.loc[family, "q_F20"] = aic.bh(t.loc[family, "p"].astype(float))
    t["q39"] = np.nan
    t.loc[IN39 + X19, "q39"] = aic.bh(t.loc[IN39 + X19, "p"].astype(float))
    return t.sort_values("p")
tv = typed.copy()
for f in X19:
    tv[f] = tv.cell_ID.astype(str).map(xv[f])
vl, vs = tv[(tv.predictCluster == 16) & tv.cellType.eq("VEN")], tv[(tv.predictCluster == 8) & tv.cellType.eq("VEN")]
t = two_group(vl, vs)
print(f"Variant, cluster 16 VENs (n {len(vl)}) against cluster 8 VENs (n {len(vs)}), lowest p first:")
print(t.head(6).to_string(float_format=lambda v: f"{v:.4g}"))
print("Features with FDR < 0.05, 39-feature family:", int((t.q39 < 0.05).sum()), "; 20 Methods features:", int((t.q_F20 < 0.05).sum()))
t = two_group(a, b)
print(f"\nVariant, expert labels VEN_Long (n {len(a)}) against VEN_Short (n {len(b)}), lowest p first:")
print(t.head(6).to_string(float_format=lambda v: f"{v:.4g}"))
print("Features with FDR < 0.05, 39-feature family:", int((t.q39 < 0.05).sum()), "; 20 Methods features:", int((t.q_F20 < 0.05).sum()))

**Result: venl blocked (data missing), for both claims.** Among the 68 VENs of the dandiset, 18 are labelled
VEN_Long and 11 VEN_Short, against the 85 VEN-L and 41 VEN-S recorded without RNA of Extended
Data Fig. 9c,d and Supplementary Table 8, which are not here. As variants that do not decide:
half-width 3.30 against 2.98 ms (p = 0.37, FDR 0.37 over the two features) and amplitude 84.7
against 86.4 mV (p = 0.36); no feature has FDR below 0.05 over the 39-feature family or the 20
Methods features, either for cluster 16 against cluster 8 VENs (lowest FR_slope, p = 0.0017, FDR
0.068) or for VEN_Long against VEN_Short (lowest FR_slope, p = 0.013, FDR 0.20). Some VEN-typed
cells carry a PC subtype label (PC_L5 4, L23_PC 3, PC_L6 1, PC_L23 1).

## supp9-venl-more-distinct-from-ct

> VEN-L differs more from L5/6 CT neurons in physiology than VEN-S does.

In [ ]:
ctc_all = tv[(tv.predictCluster == 8) & tv.cellType.eq("PC")]
# A recording filed twice with conflicting labels (cell 8) is left out: n175 (cluster 8 PC) holds the same voltage
# data as B289 (cluster 16, L5 ET).
conflict = set(cells.trace_hash[cells.trace_hash.duplicated(keep=False)])
ctc = ctc_all[~ctc_all.trace_hash.isin(conflict)]
print("VEN-L (cluster 16 VEN):", len(vl), "; VEN-S (cluster 8 VEN):", len(vs), "; L5/6 CT (cluster 8 PC):", len(ctc),
      "(legend 10); left out as a duplicate recording:", sorted(set(ctc_all.cell_ID) - set(ctc.cell_ID)))
res = {}
for ct_name, ctd in [("", ctc), (" (variant, duplicate kept)", ctc_all)]:
    for name, dfv in [("VEN-L vs L5/6 CT", vl), ("VEN-S vs L5/6 CT", vs)]:
        t = two_group(dfv, ctd)
        res[name + ct_name] = {"features FDR<0.05, 39": int((t.q39 < 0.05).sum()), "features FDR<0.05, F20": int((t.q_F20 < 0.05).sum()),
                               "median q39": round(float(t.q39.median()), 3), "median q_F20": round(float(t.q_F20.median()), 3)}
        if not ct_name:
            print(f"\n{name}, lowest p first:"); print(t.head(10).to_string(float_format=lambda v: f"{v:.4g}"))
print(pd.DataFrame(res).T.to_string())

**Result: supp9 reproduced** Against the 10 L5/6 CT PCs (legend 10; n175, a
second copy of the L5 ET recording B289, is left out), VEN-L (14) differ in 6 of the 39 features at
FDR below 0.05 (maximal firing rate and spike count, ISI Fano factor and CV, FR_slope and
firstISI_slope) and VEN-S (15) in none; the median FDR is 0.26 against 0.90. Over the 20 Methods
features the counts are 1 and 0. With n175 kept the counts are 12 and 0.

## morph-table-link

> (Link claim added by this attempt.) The rows of the paper's morphological feature table
> morphFeatMatrix.xlsx are cells of the dandiset, matched by cell ID, with the same transcriptomic
> cluster as the cell table metaInfo_ephysio_patchSeq.csv.

In [ ]:
mx = pd.read_excel(aic.MORPH_XLSX, sheet_name="morphFeatMatrix").rename(columns={"Unnamed: 0": "cell_ID"})
mx["cell_ID"] = mx.cell_ID.astype(str).str.strip("'")
mfeat = [c for c in mx.columns if c not in ("cell_ID", "cellTypeID", "cellType_expert", "predictClusterID")]
print("Rows:", len(mx), "; distinct cell IDs:", mx.cell_ID.nunique(), "; morphological features:", len(mfeat), "(Methods: 40)")
mx["filename"] = mx.cell_ID.map(m1.filename_nwb)
in_ds = mx.cell_ID.isin(cells.cell_id)
print("Rows whose cell ID is a file's cell ID in the dandiset:", int(in_ds.sum()), f"({in_ds.mean():.1%})")
mx["cluster_table"] = mx.cell_ID.map(m1.predictCluster)
both_typed = mx.predictClusterID.notna() & mx.cluster_table.notna()
print("Rows with a cluster in both tables:", int(both_typed.sum()), "; agreeing:",
      int((mx.predictClusterID[both_typed] == mx.cluster_table[both_typed]).sum()))
print("Rows marked traced in the cell table:", int(mx.cell_ID.map(m1.morphoTraced).eq("traced").sum()),
      "; rows traced in the cell table:", int(meta.morphoTraced.eq("traced").sum()))

**Result: morphlink reproduced.** All 166 rows of morphFeatMatrix.xlsx carry a cell ID that is the cell ID of a
file in the dandiset, and 163 of the 165 rows with a cluster in both tables have the same cluster
as the cell table. The table holds 57 morphological features, not the Methods' 40.**

## morph-more-distinct-than-ephys

> Morphological differences between projection neuron groups are more pronounced than their
> physiological differences, with more features differing.

The comparisons of calFDR_ephsio.m (each group against the other five, and each pair), with BH FDR
within each comparison. Morphological features from morphFeatMatrix.xlsx (all its features; the
Methods' 40 are not listed), groups from the cell table's labels; physiological features from the
dandiset's recordings (the 20 Methods features, which the criterion names, and the 39-feature family as a variant).

In [ ]:
mg = mx.merge(lab[["cell_ID", "group"]], on="cell_ID", how="left").dropna(subset=["group"])
print("Morphology cells per group:", mg.group.value_counts().reindex(aic.GROUPS).to_dict(),
      "; Fig. 2h legend: 25, 16, 54, 18, 11, 13")
comparisons = [([g], [o for o in aic.GROUPS if o != g]) for g in aic.GROUPS] + \
    [([a], [b]) for i, a in enumerate(aic.GROUPS) for b in aic.GROUPS[i + 1:]]
rows = []
for A, B in comparisons:
    xa, xb = mg[mg.group.isin(A)], mg[mg.group.isin(B)]
    pm = [stats.mannwhitneyu(xa[f].dropna(), xb[f].dropna()).pvalue if xa[f].notna().sum() > 1 and xb[f].notna().sum() > 1 else np.nan
          for f in mfeat]
    qm = aic.bh(pm)
    te = compare(grp, A, B)
    rows.append({"comparison": f"{A[0]} vs {'rest' if len(B) > 1 else B[0]}",
                 "morph sig": int(np.nansum(qm < 0.05)), "morph tested": int(np.isfinite(qm).sum()),
                 "ephys F20 sig": int((te.loc[F20, "q_F20"] < 0.05).sum()),
                 "ephys 39 sig": int((te.loc[IN39 + X19, "q39"] < 0.05).sum()),
                 "ephys 39 tested": int(te.loc[IN39 + X19, "q39"].notna().sum())})
mc = pd.DataFrame(rows)
print(mc.to_string(index=False))
tot = mc.sum(numeric_only=True)
print("\nSummed over comparisons:", tot.to_dict())
print("(a) count: morph", int(tot["morph sig"]), "against ephys F20", int(tot["ephys F20 sig"]), "; 39-feature family",
      int(tot["ephys 39 sig"]))
print("(b) fraction: morph", round(tot["morph sig"] / tot["morph tested"], 3), "against ephys F20",
      round(tot["ephys F20 sig"] / (20 * len(comparisons)), 3), "; 39-feature family",
      round(tot["ephys 39 sig"] / tot["ephys 39 tested"], 3))

# The Methods count 40 morphological features without listing them; the table has 57. Random sets of 40 of the 57.
rng = np.random.default_rng(0)
pm_all = []
for A, B in comparisons:
    xa, xb = mg[mg.group.isin(A)], mg[mg.group.isin(B)]
    pm_all.append([stats.mannwhitneyu(xa[f].dropna(), xb[f].dropna()).pvalue if xa[f].notna().sum() > 1 and xb[f].notna().sum() > 1
                   else np.nan for f in mfeat])
pm_all = np.array(pm_all)
sub = []
for _ in range(500):
    k = rng.choice(len(mfeat), 40, replace=False)
    q = np.array([aic.bh(row[k]) for row in pm_all])
    sub.append((int(np.nansum(q < 0.05)), np.nansum(q < 0.05) / np.isfinite(q).sum()))
sub = np.array(sub)
print("Variant, 500 random sets of 40 of the 57 morphological features: count from", int(sub[:, 0].min()), "to", int(sub[:, 0].max()),
      "; fraction from", round(sub[:, 1].min(), 3), "to", round(sub[:, 1].max(), 3))

**Result: morph reproduced** Summed over the 21 comparisons, 445 of 1,185
morphological tests (57 features; fraction 0.376) have FDR below 0.05, against 120 of 420
physiological tests over the 20 Methods features (fraction 0.286): more under both the count and
the fraction. Over the 39-feature physiological family the count is 150 and the fraction 0.183,
which leads to the same result. Over 500 random sets of 40 of the 57 morphological features, as the
Methods count 40 without listing them, the count runs from 255 to 372 and the fraction from 0.308
to 0.443, above the physiological values in every set.

## fig7c-tau-rna-distance

> Across pairs of projection neuron types, distances in membrane time constant correlate with
> transcriptomic distances (r = 0.53), and other morphoelectric properties show similar correlations.

The transcriptomic side is the 10x excitatory neurons of GEO GSE319557 (Ext.h5ad in
GSE319557_AIC_scRNA_h5ad.tar.gz, 24.7 GB), streamed until that member and read in memory. Its
leiden clusters are numbered as the cell table's predictCluster. Per cluster, the mean log2(x + 1)
UMI count of the 500 most variable genes (Methods, Variability analysis); the transcriptomic
distance of a pair is 1 minus the Pearson r of those means; the tau distance is the absolute
difference of the clusters' mean tau (Euclidean distance in one dimension), over the 12 clusters the
claims file lists.

In [ ]:
import itertools
t0 = time.time()
buf = aic.geo_member()
print(f"Streamed {aic.GEO_MEMBER} from GEO: {buf.getbuffer().nbytes / 1e9:.2f} GB in {time.time() - t0:.0f} s")
gmeans, glabels, gn = aic.ext_cluster_means(buf)
del buf
print("10x clusters (leiden number, label, cells):")
table_labels = typed.drop_duplicates("predictCluster").set_index("predictCluster").predictCluster_label
table_labels.index = table_labels.index.astype(int)
print(pd.DataFrame({"10x label": glabels, "10x cells": gn, "cell table label": table_labels.reindex(glabels.index)}).to_string())
T7 = [0, 1, 2, 4, 5, 6, 8, 14, 15, 16, 17, 18]
ty = typed[typed.predictCluster.isin(T7)].drop_duplicates("trace_hash")
print("\nPatch-seq cells per cluster (distinct recordings):", ty.predictCluster.value_counts().sort_index().astype(int).to_dict())
rows = []
for genes, M in gmeans.items():
    C = np.corrcoef(M.loc[T7].values)
    rna = {(a, b): 1 - C[T7.index(a), T7.index(b)] for a, b in itertools.combinations(T7, 2)}
    for feat in ["tau_methods", "tau_code", "tau_sweep_median", "max_firing_rate"]:
        mt = ty.groupby("predictCluster")[feat].mean()
        x = np.array([abs(mt[a] - mt[b]) for a, b in rna])
        y = np.array(list(rna.values()))
        r, p = stats.pearsonr(x, y)
        rows.append({"genes": genes, "feature": feat, "pairs": len(x), "r": round(r, 3), "p": float(f"{p:.3g}")})
r7 = pd.DataFrame(rows)
print("\nPearson r of feature distance against transcriptomic distance over the cluster pairs (expected 0.53, tolerance 0.053):")
print(r7.to_string(index=False))

**Result: fig7c not reproduced** Over the 66 pairs of the 12 clusters,
the Methods' tau distance correlates with the transcriptomic distance with r = 0.105 (p = 0.40),
far from 0.53 and not significant. With the code's first-sweep tau r = 0.36 (p = 0.003), positive
but outside the tolerance; with the median over sweeps r = 0.08 (p = 0.51). With the genes chosen
by variance in place of dispersion, r = -0.04 for the Methods' tau. The maximal firing rate gives
r = 0.07. Clusters 1 and 4 hold 4 and 5 Patch-seq cells.

## Summary

| Claim | Outcome |
|---|---|
| `patchseq-cell-table-link` | Reproduced |
| `morph-table-link` | Reproduced |
| `patchseq-sample` | Partially reproduced |
| `patchseq-mapping` | Partially reproduced |
| `fig2a-subclass-counts` | Partially reproduced |
| `ven-map-et-ct` | Reproduced |
| `fig6f-ven-subtype-mapping` | Partially reproduced |
| `fig2d-l23it-small-tau` | Reproduced |
| `fig2f-l23it-narrow-ap` | Partially reproduced |
| `supp11-rm-itrorb-l6pc` | Partially reproduced |
| `supp11-l5it-threshold-latency` | Reproduced |
| `supp11-ven-broad-ap` | Partially reproduced |
| `supp11-l5et-deep-sag` | Blocked: methods underspecified |
| `ven-firing-rate-variability` | Not reproduced |
| `l5et-vs-l5it-sag-rm-rate` | Partially reproduced |
| `fig3e-phase-peak-index` | Partially reproduced |
| `fig3d-saddle-phase-plane` | Partially reproduced |
| `fig4ac-ven-broader-smaller-ap` | Blocked: data missing |
| `fig4i-peak-index-ais-distance-real` | Not reproduced |
| `ext9cd-ven-subtypes-ap` | Blocked: data missing |
| `supp8-ven-subtypes-similar-features` | Blocked: data missing |
| `supp9-venl-more-distinct-from-ct` | Reproduced |
| `morph-more-distinct-than-ephys` | Reproduced |
| `fig7c-tau-rna-distance` | Not reproduced |
| nine `elsewhere` claims | Blocked: other |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The features are computed from the deposited traces
with the Methods' definitions and PatchAnaLab's details, not taken from the paper's feature table;
they match it closely for the AP measures (about 97% of cells within 10%), rheobase, latency and
the first-sweep tau. The Methods' tau (averaged over all hyperpolarizing sweeps) is about 26% longer
than the table's, which holds the code's first-sweep tau. Of the claims file's FDR family (39
features), 19 features are defined nowhere in the paper or its package; their values are taken
from the paper's feature table, as an assumption. The phase peak index is read automatically,
where the paper clicked the peaks by hand; it agrees with the hand-read values only loosely, and
the hand-read values give the paper's results for Fig. 3e and 4i where the automatic ones give
them in part or not at all. Groups come from the cell table, which differs from the paper's feature
table for two cells. The transcriptomic distances use the 500 genes of highest dispersion in the
10x data, as an assumption. No outcome rests on these two choices: the variants give the
same outcomes. Half-width and tau differ between the two amplifiers' files in every
group, in the computed values and in the paper's table, and the groups are unevenly spread over
them; the narrow APs and short tau of L2/3 IT HPCAL1 are not significant within either device.

What was found about the dataset. The files hold the Patch-seq electrophysiology and no cell
types, transcriptomic types or VEN subtypes, which come only from the paper's code repository.
Five pairs of files hold identical recordings under different cell IDs, four of them under
different subjects. The cell table names one file twice and omits another (s172). The archive
lists 10 subjects, where the paper used 12. Every session start time is the same placeholder, some
session IDs carry another subject's date, the stimulus unit is stated as amperes for values in pA,
the stimulus series have no starting time, and five files have a device name that does not fit
their sampling rate. The device follows the recording year, not the RNA status, where the Methods
put the Patch-seq recordings on the HEKA amplifier. The additional recordings of Fig. 4a-c and
Extended Data Fig. 9c,d are not in the dandiset.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")